In [1]:
#claudio-i@ace.me

In [2]:

import numpy as np
import pandas as pd
from scipy.io import loadmat
import datetime as dt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import savgol_filter
import matplotlib.ticker as mticker
from multitaper import MTSpec

In [4]:
import pickle

with open('/home/cl/Google/@doctorado-sync/database/wind-cs/cs-wind_timeseries.pkl', 'rb') as f:
  cs = pickle.load(f)

In [5]:
import numpy as np
theta = np.deg2rad(cs['wind_dir_deg'].astype(float))
# Convert speed from knots to m/s
W = cs['wind_speed_kt'].astype(float) * 0.514444
# Meteorological convention (FROM): minus signs
cs['u'] = -W * np.sin(theta)   # +u is toward east
cs['v'] = -W * np.cos(theta)   # +v is toward north
cs['speed_ms'] = cs['wind_speed_kt'].astype(float) * 0.514444
del W, theta
cs

,time,wind_dir_deg,wind_speed_kt,vrb,source_file,u,v,speed_ms
0,2004-01-01 00:00:00,20.0,9.0,False,200401.html,-1.583552,-4.350773e+00,4.629996
1,2004-01-01 01:00:00,20.0,4.0,False,200401.html,-0.703801,-1.933677e+00,2.057776
2,2004-01-01 02:00:00,20.0,4.0,False,200401.html,-0.703801,-1.933677e+00,2.057776
3,2004-01-01 03:00:00,20.0,3.0,False,200401.html,-0.527851,-1.450258e+00,1.543332
4,2004-01-01 04:00:00,90.0,3.0,False,200401.html,-1.543332,-9.450183e-17,1.543332
...,...,...,...,...,...,...,...,...
26278,2006-12-31 19:00:00,250.0,14.0,False,200612.html,6.767869,2.463303e+00,7.202216
26279,2006-12-31 20:00:00,230.0,13.0,False,200612.html,5.123131,4.298817e+00,6.687772
26280,2006-12-31 21:00:00,230.0,13.0,False,200612.html,5.123131,4.298817e+00,6.687772
26281,2006-12-31 22:00:00,240.0,19.0,False,200612.html,8.464910,4.887218e+00,9.774436


In [6]:
pwd

'/home/cl'

In [7]:
cd '/home/cl/Google/@doctorado-sync/codes'

/home/cl/Google/@doctorado-sync/codes


In [8]:
import numpy as np

def pca_time_depth(X):
    """
    X: array (ntime, ndepth). Returns PCs, EOFs, variance fraction.
    - Demeans each depth column.
    - Fills NaNs with 0 after demeaning (i.e., column mean).
    - Uses SVD on anomalies.
    """
    X = np.asarray(X, dtype=float)
    ntime, ndepth = X.shape

    # anomalies (demean columns)
    col_mean = np.nanmean(X, axis=0)
    Xa = X - col_mean

    # fill NaNs with 0 (column mean after demeaning)
    Xa = np.where(np.isnan(Xa), 0.0, Xa)

    # SVD: Xa = U S V^T
    U, S, VT = np.linalg.svd(Xa, full_matrices=False)
    V = VT.T

    # PCs (time series) and EOFs (vertical modes)
    PCs = U @ np.diag(S)          # (ntime, r)
    EOFs = V                      # (ndepth, r)

    # variance explained
    total_var = np.sum(np.var(Xa, axis=0, ddof=1))
    mode_var = (S**2) / (ntime - 1)
    var_frac = mode_var / total_var           # fraction per mode
    var_pct = 100 * var_frac

    return PCs, EOFs, var_frac, var_pct

In [9]:
import pickle

with open('/home/cl/Google/@doctorado-sync/database/adcp-l1/dataA_h.pkl', 'rb') as f:
    dataA = pickle.load(f)
with open('/home/cl/Google/@doctorado-sync/database/adcp-l1/dataB_h.pkl', 'rb') as f:
    dataB = pickle.load(f)
with open('/home/cl/Google/@doctorado-sync/database/adcp-l1/dataC_h.pkl', 'rb') as f:
    dataC = pickle.load(f)
with open('/home/cl/Google/@doctorado-sync/database/adcp-l1/dataD_h.pkl', 'rb') as f:
    dataD = pickle.load(f)
with open('/home/cl/Google/@doctorado-sync/database/adcp-l1/dataE_h.pkl', 'rb') as f:
    dataE = pickle.load(f)

In [10]:
dataA['time'][0],dataA['time'][-1]

(Timestamp('2004-12-30 17:00:00'), Timestamp('2005-01-26 12:00:00'))

In [11]:
dataE['time'][0],dataE['time'][-1]

(Timestamp('2006-03-10 19:00:00'), Timestamp('2006-05-29 02:00:00'))

In [12]:
dataC['time'][0],dataC['time'][-1]

(Timestamp('2005-07-16 17:00:00'), Timestamp('2005-09-04 15:00:00'))

In [13]:
u_PCs, u_EOFs, u_var_frac, u_var_pct = pca_time_depth(dataA['u'][:,0:22])
v_PCs, v_EOFs, v_var_frac, v_var_pct = pca_time_depth(dataA['v'][:,0:22])

for i, p in enumerate(u_var_pct[:5], start=1):
    print(f"Mode-U {i}: {p:.2f}%")
for i, p in enumerate(v_var_pct[:5], start=1):
    print(f"Mode-V {i}: {p:.2f}%")

Mode-U 1: 35.89%
Mode-U 2: 29.59%
Mode-U 3: 14.43%
Mode-U 4: 8.34%
Mode-U 5: 4.65%
Mode-V 1: 47.37%
Mode-V 2: 21.12%
Mode-V 3: 14.73%
Mode-V 4: 8.13%
Mode-V 5: 2.92%


In [14]:
k = 10
X_recon = u_PCs[:, :k] @ u_EOFs[:, :k].T  # anomalies
X_recon += np.nanmean(dataA['u'][:,0:22], axis=0)
dataA['u_re'] = X_recon
del X_recon

k = 10
X_recon = v_PCs[:, :k] @ v_EOFs[:, :k].T  # anomalies
X_recon += np.nanmean(dataA['v'][:,0:22], axis=0)
dataA['v_re'] = X_recon
del X_recon

In [15]:
u_PCs, u_EOFs, u_var_frac, u_var_pct = pca_time_depth(dataE['u'][:,0:44])
v_PCs, v_EOFs, v_var_frac, v_var_pct = pca_time_depth(dataE['v'][:,0:44])

for i, p in enumerate(u_var_pct[:5], start=1):
    print(f"Mode-U {i}: {p:.2f}%")
for i, p in enumerate(v_var_pct[:5], start=1):
    print(f"Mode-V {i}: {p:.2f}%")

Mode-U 1: 35.71%
Mode-U 2: 14.01%
Mode-U 3: 11.54%
Mode-U 4: 5.48%
Mode-U 5: 4.98%
Mode-V 1: 34.64%
Mode-V 2: 28.24%
Mode-V 3: 8.47%
Mode-V 4: 4.44%
Mode-V 5: 3.66%


In [16]:
k = 10
X_recon = u_PCs[:, :k] @ u_EOFs[:, :k].T  # anomalies
X_recon += np.nanmean(dataE['u'][:,0:44], axis=0)
dataE['u_re'] = X_recon
del X_recon

k = 10
X_recon = v_PCs[:, :k] @ v_EOFs[:, :k].T  # anomalies
X_recon += np.nanmean(dataE['v'][:,0:44], axis=0)
dataE['v_re'] = X_recon
del X_recon

In [17]:
u_PCs, u_EOFs, u_var_frac, u_var_pct = pca_time_depth(dataC['u'][:,0:44])
v_PCs, v_EOFs, v_var_frac, v_var_pct = pca_time_depth(dataC['v'][:,0:44])

for i, p in enumerate(u_var_pct[:5], start=1):
    print(f"Mode-U {i}: {p:.2f}%")
for i, p in enumerate(v_var_pct[:5], start=1):
    print(f"Mode-V {i}: {p:.2f}%")

Mode-U 1: 40.57%
Mode-U 2: 22.25%
Mode-U 3: 8.55%
Mode-U 4: 6.37%
Mode-U 5: 4.28%
Mode-V 1: 56.66%
Mode-V 2: 21.59%
Mode-V 3: 6.45%
Mode-V 4: 3.35%
Mode-V 5: 2.66%


In [18]:
k = 10
X_recon = u_PCs[:, :k] @ u_EOFs[:, :k].T  # anomalies
X_recon += np.nanmean(dataC['u'][:,0:44], axis=0)
dataC['u_re'] = X_recon
del X_recon

k = 10
X_recon = v_PCs[:, :k] @ v_EOFs[:, :k].T  # anomalies
X_recon += np.nanmean(dataC['v'][:,0:44], axis=0)
dataC['v_re'] = X_recon
del X_recon

In [19]:
dataA['depth'] = dataA['depth'][0:22]
dataE['depth'] = dataE['depth'][0:44]
dataC['depth'] = dataC['depth'][0:44]

In [20]:
u_re1 = dataA['u_re']
v_re1 = dataA['v_re']
t1 = dataA['time']

#----------

u_re2, u_re3, u_re4 = np.array_split(dataE['u_re'], 3)
v_re2, v_re3, v_re4 = np.array_split(dataE['v_re'], 3)
t2, t3, t4 = np.array_split(dataE['time'], 3)

#----------

u_re5, u_re6 = np.array_split(dataC['u_re'], 2)
v_re5, v_re6 = np.array_split(dataC['v_re'], 2)
t5, t6 = np.array_split(dataC['time'], 2)

In [21]:
from matplotlib import pyplot
depthT = [-1 * x for x in dataA['depth'][0:20]]
pyplot.rcParams['figure.dpi'] = 200
pyplot.rcParams['savefig.dpi'] = 200

import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

dt = 30*60
for i in range(0,20):
    x = np.cumsum(savgol_filter(u_re1[:,i],24,2)/1000) * dt  # Eastward displacement (m)
    y = np.cumsum(savgol_filter(v_re1[:,i],24,2)/1000) * dt  # Northward displacement (m)
    x_km = x / 1000
    y_km = y / 1000
    z = np.full_like(x_km, depthT[i])

    fig.add_trace(go.Scatter3d(
        x=x_km, y=y_km, z=z,
        mode='lines',
        name=f"Trajectory {depthT[i]}"
    ))

# Layout settings
fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Depth"
    )
)

fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="Eastward displacement (km)",
        yaxis_title="Northward displacement (km)",
        zaxis=dict(
            title="Depth (mab)",
            autorange="reversed"   # Invert the z-axis
        )
    ),
    width=1200,
    height=800
)


fig.show()


In [22]:
dataC['depth'][0:44]

array([ 4.21999979,  6.21999979,  8.21999979, 10.21999979, 12.21999979,
       14.21999979, 16.21999979, 18.21999979, 20.21999979, 22.21999979,
       24.21999979, 26.21999979, 28.21999979, 30.21999979, 32.21999979,
       34.21999979, 36.21999979, 38.21999979, 40.21999979, 42.21999979,
       44.21999979, 46.21999979, 48.21999979, 50.21999979, 52.21999979,
       54.21999979, 56.21999979, 58.21999979, 60.21999979, 62.21999979,
       64.21999979, 66.21999979, 68.21999979, 70.21999979, 72.21999979,
       74.21999979, 76.21999979, 78.21999979, 80.21999979, 82.21999979,
       84.21999979, 86.21999979, 88.21999979, 90.21999979])

In [23]:
from matplotlib import pyplot
depthT = [-1 * x for x in dataC['depth'][0:44]]
pyplot.rcParams['figure.dpi'] = 200
pyplot.rcParams['savefig.dpi'] = 200

import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

dt = 30*60
for i in range(0,44):
    x = np.cumsum(savgol_filter(u_re2[:,i],24,2)/1000) * dt  # Eastward displacement (m)
    y = np.cumsum(savgol_filter(v_re2[:,i],24,2)/1000) * dt  # Northward displacement (m)
    x_km = x / 1000
    y_km = y / 1000
    z = np.full_like(x_km, depthT[i])

    fig.add_trace(go.Scatter3d(
        x=x_km, y=y_km, z=z,
        mode='lines',
        name=f"Trajectory {np.round(depthT[i])}"
    ))

# Layout settings
fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Depth"
    )
)

fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="Eastward displacement (km)",
        yaxis_title="Northward displacement (km)",
        zaxis=dict(
            title="Depth (mab)",
            autorange="reversed"   # Invert the z-axis
        )
    ),
    width=1200,
    height=800
)


fig.show()

In [24]:
from matplotlib import pyplot
depthT = [-1 * x for x in dataC['depth'][0:44]]
pyplot.rcParams['figure.dpi'] = 200
pyplot.rcParams['savefig.dpi'] = 200

import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

dt = 30*60
for i in range(0,44):
    x = np.cumsum(savgol_filter(u_re3[:,i],24,2)/1000) * dt  # Eastward displacement (m)
    y = np.cumsum(savgol_filter(v_re3[:,i],24,2)/1000) * dt  # Northward displacement (m)
    x_km = x / 1000
    y_km = y / 1000
    z = np.full_like(x_km, depthT[i])

    fig.add_trace(go.Scatter3d(
        x=x_km, y=y_km, z=z,
        mode='lines',
        name=f"Trajectory {depthT[i]}"
    ))

# Layout settings
fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Depth"
    )
)

fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="Eastward displacement (km)",
        yaxis_title="Northward displacement (km)",
        zaxis=dict(
            title="Depth (mab)",
            autorange="reversed"   # Invert the z-axis
        )
    ),
    width=1200,
    height=800
)


fig.show()

In [25]:
from matplotlib import pyplot
depthT = [-1 * x for x in dataC['depth'][0:44]]
pyplot.rcParams['figure.dpi'] = 200
pyplot.rcParams['savefig.dpi'] = 200

import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

dt = 30*60
for i in range(0,44):
    x = np.cumsum(savgol_filter(u_re4[:,i],24,2)/1000) * dt  # Eastward displacement (m)
    y = np.cumsum(savgol_filter(v_re4[:,i],24,2)/1000) * dt  # Northward displacement (m)
    x_km = x / 1000
    y_km = y / 1000
    z = np.full_like(x_km, depthT[i])

    fig.add_trace(go.Scatter3d(
        x=x_km, y=y_km, z=z,
        mode='lines',
        name=f"Trajectory {depthT[i]}"
    ))

# Layout settings
fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Depth"
    )
)

fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="Eastward displacement (km)",
        yaxis_title="Northward displacement (km)",
        zaxis=dict(
            title="Depth (mab)",
            autorange="reversed"   # Invert the z-axis
        )
    ),
    width=1200,
    height=800
)


fig.show()

In [26]:
from matplotlib import pyplot
depthT = [-1 * x for x in dataC['depth'][0:44]]
pyplot.rcParams['figure.dpi'] = 200
pyplot.rcParams['savefig.dpi'] = 200

import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

dt = 30*60
for i in range(0,44):
    x = np.cumsum(savgol_filter(u_re5[:,i],24,2)/1000) * dt  # Eastward displacement (m)
    y = np.cumsum(savgol_filter(v_re5[:,i],24,2)/1000) * dt  # Northward displacement (m)
    x_km = x / 1000
    y_km = y / 1000
    z = np.full_like(x_km, depthT[i])

    fig.add_trace(go.Scatter3d(
        x=x_km, y=y_km, z=z,
        mode='lines',
        name=f"Trajectory {depthT[i]}"
    ))

# Layout settings
fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Depth"
    )
)

fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="Eastward displacement (km)",
        yaxis_title="Northward displacement (km)",
        zaxis=dict(
            title="Depth (mab)",
            autorange="reversed"   # Invert the z-axis
        )
    ),
    width=1200,
    height=800
)


fig.show()

In [27]:
from matplotlib import pyplot
depthT = [-1 * x for x in dataC['depth'][0:44]]
pyplot.rcParams['figure.dpi'] = 200
pyplot.rcParams['savefig.dpi'] = 200

import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

dt = 30*60
for i in range(0,44):
    x = np.cumsum(savgol_filter(u_re6[:,i],24,2)/1000) * dt  # Eastward displacement (m)
    y = np.cumsum(savgol_filter(v_re6[:,i],24,2)/1000) * dt  # Northward displacement (m)
    x_km = x / 1000
    y_km = y / 1000
    z = np.full_like(x_km, depthT[i])

    fig.add_trace(go.Scatter3d(
        x=x_km, y=y_km, z=z,
        mode='lines',
        name=f"Trajectory {depthT[i]}"
    ))

# Layout settings
fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Depth"
    )
)

fig.update_layout(
    title="Multiple Trajectories at Different Depths",
    scene=dict(
        xaxis_title="Eastward displacement (km)",
        yaxis_title="Northward displacement (km)",
        zaxis=dict(
            title="Depth (mab)",
            autorange="reversed"   # Invert the z-axis
        )
    ),
    width=1200,
    height=800
)


fig.show()